# Day 21 — `argparse`: Build a Real CLI

> ⚠️ **Why this matters.** Your english-helper has a REPL (read-eval-print loop). REPLs are great for interactive use. But `english-helper add ubiquitous` from the shell is even better — composable with other Unix tools, scriptable, no startup ceremony. That's what `argparse` gets you.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/21-argparse.ipynb)

## What you'll do today

- [ ] You can design a CLI with subcommands
- [ ] You know `argparse` basics: positional, optional, flags, subparsers
- [ ] You can write `--help` text users won't hate
- [ ] english-helper has both a REPL AND a one-shot CLI mode

## 1. Minimal argparse

In [ ]:
import argparse

def main():
    parser = argparse.ArgumentParser(prog='english-helper', description='Study English vocabulary.')
    parser.add_argument('word', help='word to look up')
    parser.add_argument('--limit', type=int, default=10)
    parser.add_argument('-v', '--verbose', action='store_true')
    args = parser.parse_args()
    print(args)

# usage:
# $ english-helper thorough --limit 5 -v
# Namespace(word='thorough', limit=5, verbose=True)

**Anatomy:**

- `parser.add_argument('word')` — positional, required
- `parser.add_argument('--limit', type=int)` — named, typed
- `parser.add_argument('-v', '--verbose', action='store_true')` — boolean flag
- `parser.parse_args()` — parses sys.argv, returns Namespace

If user makes a mistake, argparse prints a friendly error message and exits with code 2.

## 2. Subcommands

In [ ]:
import argparse

def cmd_add(args):
    print(f'Adding {args.word}')

def cmd_lookup(args):
    print(f'Looking up {args.word}')

def cmd_list(args):
    print('Listing all words')

def main():
    parser = argparse.ArgumentParser(prog='english-helper')
    sub = parser.add_subparsers(dest='cmd', required=True)
    
    p_add = sub.add_parser('add', help='Add a word')
    p_add.add_argument('word')
    p_add.set_defaults(func=cmd_add)
    
    p_lookup = sub.add_parser('lookup', help='Look up a word')
    p_lookup.add_argument('word')
    p_lookup.set_defaults(func=cmd_lookup)
    
    p_list = sub.add_parser('list', help='List all words')
    p_list.set_defaults(func=cmd_list)
    
    args = parser.parse_args()
    args.func(args)

Now: `english-helper add ubiquitous`, `english-helper lookup thorough`, `english-helper list`.

**Pattern:**
- Each subcommand gets its own parser
- Each calls `set_defaults(func=...)` to bind a handler function
- Top-level reads `args.func` and calls it

## 3. Designing a good `--help`

Run `english-helper --help`:

```
usage: english-helper [-h] {add,lookup,list,quiz} ...

Study English vocabulary.

positional arguments:
  {add,lookup,list,quiz}
    add                 Add a word
    lookup              Look up a word
    list                List all words
    quiz                Run a pronunciation quiz

options:
  -h, --help            show this help message and exit
```

**This is your tool's documentation.** Spend time on the descriptions. A confused user will reach for `--help` first.

**Tips:**

- One-line `help=` for each arg (shows in help)
- Use `description=` on the parser (shows above args)
- Use `epilog=` for examples at the bottom
- Set `type=int` (or others) so argparse validates and converts

## 4. Combining REPL + CLI

Best of both:

- `english-helper` (no args) → enters REPL
- `english-helper add thorough` → runs once and exits

In [ ]:
def main():
    parser = argparse.ArgumentParser(prog='english-helper')
    sub = parser.add_subparsers(dest='cmd')
    # ... add subparsers as before
    
    args = parser.parse_args()
    
    if args.cmd is None:
        # No subcommand → start REPL
        run_repl()
    else:
        # One-shot mode
        args.func(args)

## End-of-day mini-project — `cli.py` upgraded

> 🎯 Refactor english-helper's CLI:

1. Build a top-level argparse parser with these subcommands:
   - `add WORD`
   - `lookup WORD`
   - `remove WORD`
   - `list`
   - `quiz [--rounds N] [--mode random|sequential]`
   - `due`
   - `review`
   - `thai WORD <translation>`
   - `cache stats` / `cache clear`
2. `english-helper` (no args) drops into the REPL mode.
3. `english-helper add foo` runs once and exits.
4. Each subcommand's handler is a thin wrapper around your existing `cmd_*` functions.
5. Every subcommand has good `--help` text.

Acceptance: `english-helper --help` reads like a real tool. Examples work end-to-end.

## Connect to the project

> 🎯 **Tomorrow (Day 22):** packaging — your tool becomes globally installable via `uv tool install`.

**Quiz:** [21-argparse-quiz.ipynb](21-argparse-quiz.ipynb)